# 11 Classification Evaluation

## Purpose

This notebook runs the final locked test-set evaluation for the glycan subtype classifier.

The main idea here is:
- load one saved classifier from notebook 10
- load the saved validation-selected threshold
- run the test set once
- save final test metrics and prediction tables
- save per-label ROC and PR summaries for all labels
- plot ROC and monotonic PR curves for the top 10 labels by support

## Inputs

- one saved classifier `best_model/` folder from notebook 10
- `best_threshold.json` from the matching classifier run
- `test_classification.csv`
- `label_vocabulary.csv`

## Outputs

- `test_metrics.csv`
- `test_metrics.json`
- `per_label_metrics.csv`
- `roc_auc_per_label.csv`
- `average_precision_per_label.csv`
- `curve_aggregate_summary.csv`
- `top10_supported_roc_curves.png`
- `top10_supported_pr_curves.png`
- `test_prediction_table.csv`

## Notes to myself

This notebook is supposed to be the final reporting step, not another tuning loop. I want to keep the threshold fixed and just describe what the classifier does on the held-out test split.

## Setup note

Same pattern again.

- code stays in GitHub
- saved classifier artifacts stay in Drive
- Colab pulls the repo at the start
- the threshold from notebook 10 stays fixed here

The whole point is to keep test-set evaluation separate from model selection.

In [ ]:
# ==============================================================================
# 0. SET UP THE COLAB ENVIRONMENT
# ==============================================================================
import os
import sys

from google.colab import drive

drive.mount('/content/drive')

from tqdm.std import tqdm as plain_tqdm
import tqdm.auto as tqdm_auto
tqdm_auto.tqdm = plain_tqdm
try:
    import tqdm.notebook as tqdm_notebook
    tqdm_notebook.tqdm = plain_tqdm
except Exception:
    pass

GITHUB_OWNER = 'hb791-dev'
REPO_NAME = 'glycan-roberta'
REPO_URL = f'https://github.com/{GITHUB_OWNER}/{REPO_NAME}.git'
REPO_DIR = f'/content/{REPO_NAME}'

if not os.path.exists(REPO_DIR):
    print('Cloning repository...')
    !git clone -q {REPO_URL} {REPO_DIR}
else:
    print(f'Reusing existing repo at {REPO_DIR}')
    !git -C {REPO_DIR} pull --ff-only

%cd {REPO_DIR}

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f'Active repo directory: {REPO_DIR}')

## Choose the saved classifier run to evaluate

This is the main configuration spot.

I want the tokenizer family, pretrained experiment, source classifier run, and evaluation output folder to all stay explicit so I do not accidentally overwrite earlier evaluation results.


In [ ]:
# ==============================================================================
# 1. IMPORT HELPERS AND DEFINE THE EVALUATION RUN
# ==============================================================================
from pathlib import Path
import json

import pandas as pd
import torch
from IPython.display import Image, display

from src.classification_evaluation import (
    build_classification_evaluation_output_paths,
    build_curve_aggregate_summary,
    build_exact_match_curve_summary,
    build_multilabel_pr_summary,
    build_multilabel_roc_summary,
    build_support_weighted_error_summary,
    build_test_classification_dataset,
    compute_exact_match_confidence_scores,
    compute_exact_match_flags,
    compute_per_label_metrics,
    load_best_threshold,
    load_classifier_artifacts,
    load_test_classification_table,
    plot_exact_match_monotonic_pr_curve,
    plot_exact_match_roc_curve,
    plot_selected_monotonic_pr_curves,
    plot_selected_roc_curves,
    run_classifier_predictions,
    save_classification_evaluation_outputs,
    select_top_supported_labels,
)
from src.classification_training import (
    binarize_multilabel_predictions,
    build_classification_prediction_table,
    compute_multilabel_metrics,
)

DRIVE_ROOT = Path('/content/drive/MyDrive/ProjectRoot')
CLASSIFICATION_PREP_DIR = DRIVE_ROOT / 'results' / 'classification_prep'

TOKENIZER_FAMILY = 'manual'
PRETRAIN_EXPERIMENT_NAME = 'mlm15_L6_H512_A8_lr00001_ep100_setv1_train_only'

# This is the already-saved classifier run I want to load. Right now this is
# pointing at the random-init run that was accidentally saved under the old
# generic label.
SOURCE_CLASSIFIER_RUN_LABEL = 'cls_lr2e-5_ep10_bs16_randominit'

# Save evaluation outputs to a separate folder so I do not overwrite older
# evaluation results even when I am loading from an old classifier folder.
EVALUATION_RUN_LABEL = 'cls_lr2e-5_ep10_bs16_randominit_eval'
ALLOW_OVERWRITE_EXISTING_EVALUATION = False

MAX_LENGTH = 130
EVAL_BATCH_SIZE = 16
TOP_K_PLOTTED_LABELS = 10
MIN_SUPPORT_FOR_SUPPORT_WEIGHTED_ERROR = 1

CLASSIFIER_BEST_MODEL_DIR = (
    DRIVE_ROOT
    / 'checkpoints'
    / 'classification'
    / TOKENIZER_FAMILY
    / PRETRAIN_EXPERIMENT_NAME
    / SOURCE_CLASSIFIER_RUN_LABEL
    / 'best_model'
)
BEST_THRESHOLD_PATH = (
    DRIVE_ROOT
    / 'results'
    / 'classification_finetuning'
    / TOKENIZER_FAMILY
    / PRETRAIN_EXPERIMENT_NAME
    / SOURCE_CLASSIFIER_RUN_LABEL
    / 'best_threshold.json'
)
LABEL_VOCABULARY_PATH = (
    DRIVE_ROOT
    / 'results'
    / 'classification_finetuning'
    / TOKENIZER_FAMILY
    / PRETRAIN_EXPERIMENT_NAME
    / SOURCE_CLASSIFIER_RUN_LABEL
    / 'label_vocabulary_snapshot.csv'
)
TEST_CLASSIFICATION_PATH = CLASSIFICATION_PREP_DIR / 'test_classification.csv'

OUTPUT_PATHS = build_classification_evaluation_output_paths(
    project_root=DRIVE_ROOT,
    tokenizer_family=TOKENIZER_FAMILY,
    experiment_name=PRETRAIN_EXPERIMENT_NAME,
    classifier_run_label=EVALUATION_RUN_LABEL,
)

existing_evaluation_markers = [
    Path(OUTPUT_PATHS['evaluation_config_path']),
    Path(OUTPUT_PATHS['test_metrics_csv_path']),
    Path(OUTPUT_PATHS['exact_match_summary_path']),
]
if not ALLOW_OVERWRITE_EXISTING_EVALUATION and any(path.exists() for path in existing_evaluation_markers):
    raise FileExistsError(
        'This evaluation run label already has saved outputs. Change EVALUATION_RUN_LABEL '
        'or set ALLOW_OVERWRITE_EXISTING_EVALUATION = True if you really want to overwrite it.'
    )

evaluation_config = {
    'tokenizer_family': TOKENIZER_FAMILY,
    'pretrain_experiment_name': PRETRAIN_EXPERIMENT_NAME,
    'source_classifier_run_label': SOURCE_CLASSIFIER_RUN_LABEL,
    'evaluation_run_label': EVALUATION_RUN_LABEL,
    'allow_overwrite_existing_evaluation': ALLOW_OVERWRITE_EXISTING_EVALUATION,
    'classifier_best_model_dir': str(CLASSIFIER_BEST_MODEL_DIR),
    'best_threshold_path': str(BEST_THRESHOLD_PATH),
    'label_vocabulary_path': str(LABEL_VOCABULARY_PATH),
    'test_classification_path': str(TEST_CLASSIFICATION_PATH),
    'max_length': MAX_LENGTH,
    'eval_batch_size': EVAL_BATCH_SIZE,
    'top_k_plotted_labels': TOP_K_PLOTTED_LABELS,
    'min_support_for_support_weighted_error': MIN_SUPPORT_FOR_SUPPORT_WEIGHTED_ERROR,
}
Path(OUTPUT_PATHS['evaluation_config_path']).write_text(
    json.dumps(evaluation_config, indent=2),
    encoding='utf-8',
)

print(f'Source classifier run label: {SOURCE_CLASSIFIER_RUN_LABEL}')
print(f'Evaluation run label: {EVALUATION_RUN_LABEL}')
print(f'Classifier best-model dir: {CLASSIFIER_BEST_MODEL_DIR}')
print(f'Best threshold path: {BEST_THRESHOLD_PATH}')
print(f'Evaluation results dir: {OUTPUT_PATHS["results_dir"]}')


## Load the saved classifier, threshold, and test table

This is the point where notebook 11 becomes the locked final evaluation step.

The threshold comes from validation, not from the test set.

In [ ]:
# ==============================================================================
# 2. LOAD FINAL EVALUATION INPUTS
# ==============================================================================
artifact_bundle = load_classifier_artifacts(
    model_dir=CLASSIFIER_BEST_MODEL_DIR,
    label_vocabulary_path=LABEL_VOCABULARY_PATH,
)
model = artifact_bundle['model']
tokenizer = artifact_bundle['tokenizer']
runtime_device = artifact_bundle['runtime_device']
label_vocabulary_df = artifact_bundle['label_vocabulary_df']
label_name_to_id = artifact_bundle['label_name_to_id']

best_threshold_row = load_best_threshold(BEST_THRESHOLD_PATH)
chosen_threshold = float(best_threshold_row['threshold'])

test_df = load_test_classification_table(TEST_CLASSIFICATION_PATH)

print(f'Runtime device: {runtime_device}')
print(f'Chosen threshold: {chosen_threshold:.2f}')
print(f'Test rows: {len(test_df)}')
print(f'Label vocabulary size: {len(label_vocabulary_df)}')

## Build the tokenized test dataset

This is just the test-side version of the dataset-building step from notebook 10.

In [ ]:
# ==============================================================================
# 3. BUILD THE TEST DATASET
# ==============================================================================
test_dataset_bundle = build_test_classification_dataset(
    test_df=test_df,
    tokenizer=tokenizer,
    label_name_to_id=label_name_to_id,
    max_length=MAX_LENGTH,
)

encoded_test_df = test_dataset_bundle['test_df']
test_dataset = test_dataset_bundle['test_dataset']

print(f'Test dataset rows: {len(test_dataset)}')

## Run final test-set predictions

This is the core test-set inference step.

At this point the model, threshold, and label vocabulary are already fixed.

In [ ]:
# ==============================================================================
# 4. RUN THE FINAL TEST-SET PREDICTIONS
# ==============================================================================
prediction_bundle = run_classifier_predictions(
    model=model,
    evaluation_dataset=test_dataset,
    runtime_device=runtime_device,
    batch_size=EVAL_BATCH_SIZE,
)

test_true_labels = prediction_bundle['true_labels']
test_probabilities = prediction_bundle['probabilities']
test_binary_predictions = binarize_multilabel_predictions(
    test_probabilities,
    threshold=chosen_threshold,
)

# For the glycan-level exact-match analysis, I want one binary target
# (fully correct vs not fully correct) and one confidence score per glycan.
exact_match_flags = compute_exact_match_flags(
    true_labels=test_true_labels,
    predicted_labels=test_binary_predictions,
)
exact_match_confidence_scores = compute_exact_match_confidence_scores(
    predicted_probabilities=test_probabilities,
    threshold=chosen_threshold,
)

print('Test predictions complete.')


## Save the main final metrics

This is the compact summary I would start from if I were writing the final results section.

In [ ]:
# ==============================================================================
# 5. COMPUTE THE MAIN TEST-SET METRICS
# ==============================================================================
test_metrics = compute_multilabel_metrics(
    true_labels=test_true_labels,
    predicted_labels=test_binary_predictions,
    predicted_probabilities=test_probabilities,
)
test_metrics['threshold'] = chosen_threshold

display(pd.DataFrame([test_metrics]))

## Build per-label summaries for all labels

I want all 41 labels saved in tables even though the plots will only show the top-supported ones.

I am also making a simple support-aware weak-label table using support x (1 - F1) so I can flag labels that are both imperfect and common enough to matter.


In [ ]:
# ==============================================================================
# 6. BUILD ALL-LABEL TABLES
# ==============================================================================
per_label_metrics_df = compute_per_label_metrics(
    true_labels=test_true_labels,
    predicted_labels=test_binary_predictions,
    label_vocabulary_df=label_vocabulary_df,
)
support_weighted_error_summary_df = build_support_weighted_error_summary(
    per_label_metrics_df=per_label_metrics_df,
    min_support=MIN_SUPPORT_FOR_SUPPORT_WEIGHTED_ERROR,
)
roc_summary_df = build_multilabel_roc_summary(
    true_labels=test_true_labels,
    predicted_probabilities=test_probabilities,
    label_vocabulary_df=label_vocabulary_df,
)
pr_summary_df = build_multilabel_pr_summary(
    true_labels=test_true_labels,
    predicted_probabilities=test_probabilities,
    label_vocabulary_df=label_vocabulary_df,
)
curve_aggregate_summary_df = build_curve_aggregate_summary(
    roc_summary_df=roc_summary_df,
    pr_summary_df=pr_summary_df,
)

display(per_label_metrics_df.head(10))
display(support_weighted_error_summary_df.head(10))
display(curve_aggregate_summary_df)


## Plot label-level and exact-match curves

I still want the top-supported label plots for a first pass, but I also want one glycan-level exact-match view.

That exact-match view treats each glycan as either fully correct or not fully correct, then uses a simple confidence score based on the smallest margin from the decision threshold across all labels.


In [ ]:
# ==============================================================================
# 7. PLOT THE TOP-SUPPORTED LABELS AND THE EXACT-MATCH VIEW
# ==============================================================================
top_supported_label_names = select_top_supported_labels(
    label_summary_df=per_label_metrics_df,
    top_k=TOP_K_PLOTTED_LABELS,
)

top10_roc_summary_df = plot_selected_roc_curves(
    true_labels=test_true_labels,
    predicted_probabilities=test_probabilities,
    label_vocabulary_df=label_vocabulary_df,
    selected_label_names=top_supported_label_names,
    save_path=OUTPUT_PATHS['top10_roc_plot_path'],
)
top10_pr_summary_df = plot_selected_monotonic_pr_curves(
    true_labels=test_true_labels,
    predicted_probabilities=test_probabilities,
    label_vocabulary_df=label_vocabulary_df,
    selected_label_names=top_supported_label_names,
    save_path=OUTPUT_PATHS['top10_pr_plot_path'],
)

exact_match_summary_df = build_exact_match_curve_summary(
    exact_match_flags=exact_match_flags,
    exact_match_confidence_scores=exact_match_confidence_scores,
    threshold=chosen_threshold,
)
plot_exact_match_roc_curve(
    exact_match_flags=exact_match_flags,
    exact_match_confidence_scores=exact_match_confidence_scores,
    save_path=OUTPUT_PATHS['exact_match_roc_plot_path'],
)
plot_exact_match_monotonic_pr_curve(
    exact_match_flags=exact_match_flags,
    exact_match_confidence_scores=exact_match_confidence_scores,
    save_path=OUTPUT_PATHS['exact_match_pr_plot_path'],
)

display(top10_roc_summary_df)
display(top10_pr_summary_df)
display(exact_match_summary_df)


## Save a human-readable test prediction table

This is the table I would want if I were trying to review individual glycans after the run.

I am also adding a simple exact-match flag and exact-match confidence score so glycan-level review is easier later.


In [ ]:
# ==============================================================================
# 8. SAVE THE TEST PREDICTION TABLE
# ==============================================================================
test_prediction_table_df = build_classification_prediction_table(
    source_df=encoded_test_df,
    probabilities=test_probabilities,
    predicted_labels=test_binary_predictions,
    label_vocabulary_df=label_vocabulary_df,
)

# These two columns make it easier to sort glycans by full-set correctness later.
test_prediction_table_df['is_exact_match'] = exact_match_flags.astype(int)
test_prediction_table_df['exact_match_confidence'] = exact_match_confidence_scores

display(test_prediction_table_df.head(10))


## Save everything to Drive

This is the last writeout step so notebook 11 leaves behind one complete evaluation folder.

In [ ]:
# ==============================================================================
# 9. SAVE FINAL EVALUATION OUTPUTS
# ==============================================================================
save_classification_evaluation_outputs(
    test_metrics=test_metrics,
    per_label_metrics_df=per_label_metrics_df,
    support_weighted_error_summary_df=support_weighted_error_summary_df,
    roc_summary_df=roc_summary_df,
    pr_summary_df=pr_summary_df,
    curve_aggregate_summary_df=curve_aggregate_summary_df,
    exact_match_summary_df=exact_match_summary_df,
    top10_roc_summary_df=top10_roc_summary_df,
    top10_pr_summary_df=top10_pr_summary_df,
    test_prediction_table_df=test_prediction_table_df,
    output_paths=OUTPUT_PATHS,
)

print('Saved final evaluation files:')
for output_name, output_path in OUTPUT_PATHS.items():
    print(f'- {output_name}: {output_path}')


## Final note

At this point I should have:
- the final test metrics
- the per-label tables for all subtype labels
- a support-aware weak-label summary table
- readable top-10 ROC and PR plots
- one glycan-level exact-match ROC plot and PR plot
- a test prediction table for glycan-level review

If we need different plotted labels later, I can regenerate those without changing the locked test predictions themselves.
